In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
import default_risk.config as cfg
import xgboost as xgb
from dotenv import load_dotenv
import dtale

installments_payment_df = pd.read_parquet(cfg.CLEANS_DIR / "installments_payments.train-cleaned.parquet")

column_order_reference="days_instalment"

installments_payment_df.sort_values(["id_prev",column_order_reference,"days_instalment"],inplace=True)

data_frame_size=len(installments_payment_df)

installments_payment_df.head()

def get_time_window_metrics(days_since, installments_payment_df) :    
    last_year= installments_payment_df[installments_payment_df["days_instalment"] > days_since]

    days_since = days_since * -1

    last_year_agg= last_year.groupby("id_curr").agg({
        "amt_instalment": ["max", "min","mean","sum"],
        "amt_payment": ["max", "min","mean","sum","std"],
        "days_of_delinquency":["mean","sum"],
        "days_in_advance":["mean","sum"],
        "diff_expected_received": ["max", "min", "mean", "sum"],
        "is_delinquency" : ["mean","sum"],
        "is_underpayment" : ["mean"],
        "extra_instalament":["sum","mean"],
        })


    last_year_agg.columns = [
    f"last_{days_since}_instalments_{col[0]}_{col[1]}"
    for col in last_year_agg.columns
    ]

    last_year_agg= last_year_agg.reset_index()

    last_year_agg[f"last_{days_since}_instalments_completion_ratio"] = np.where( last_year_agg[f"last_{days_since}_instalments_amt_instalment_sum"] > 0, last_year_agg[f"last_{days_since}_instalments_amt_payment_sum"] / last_year_agg[f"last_{days_since}_instalments_amt_instalment_sum"], 1.0 ) 

    last_year_agg.to_parquet(cfg.PROCESSED_DIR / "installments_payments_last_year_metrics.parquet")
    return last_year_agg



In [7]:
installments_payment_df = pd.read_parquet(cfg.SPLITS_DIR / "installments_payments.train.parquet")
installments_payment_df.head()


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [ ]:
#we starting catching this because we are probably cutting some parts of the temporal sequence
installments_payment_df["raw_size_serie"]= installments_payment_df.groupby("id_prev").transform("size")
installments_payment_df["amount_of_versions_in_sequence"] = installments_payment_df.groupby("id_prev")["num_instalment_version"].transform("nunique")

In [ ]:
#we gonna use the knowlege recolected in the EDA. 
#For more details look eda_installments_payments.ipynb decisions summary 1#

#creating auxiliar columns
installments_payment_df["next_payment_value"] = installments_payment_df.groupby("id_prev")["days_entry_payment"].shift(-1)
nan_amount= installments_payment_df.groupby("id_prev")["days_entry_payment_is_missing"].transform("sum")

#defining the mask to separate the differents cases of missings values 
have_missing_entry_payment_mask= (installments_payment_df["next_payment_value"].isna())
not_a_deadtail_mask= (installments_payment_df["days_entry_payment"].isna() ) & ( installments_payment_df["next_payment_value"].notna())

#we want to catch the cases of  dead-tail so we starting filtering that cases with nans but that are not dead tails
non_dead_tail_nans= installments_payment_df[not_a_deadtail_mask]
ids_with_nulls_that_are_non_deadtails= non_dead_tail_nans["id_prev"].unique()
excluding_nans_non_deadtails_mask= ~(installments_payment_df["id_prev"].isin(ids_with_nulls_that_are_non_deadtails))
series_without_problematic_nans = installments_payment_df[excluding_nans_non_deadtails_mask]

#now this ID are series where the nans are deadtails, so count nans in "days_entry_payment" o "amt_payment" is calculate
#the lenght of the deadtail and we save that value in a new column
ids_with_deadtails= series_without_problematic_nans[series_without_problematic_nans["days_entry_payment"].isna()]["id_prev"].unique()
installments_payment_df["dead_tail_length"]= np.where(installments_payment_df["id_prev"].isin(ids_with_deadtails),  nan_amount, 0)

In [ ]:
#also in the decision summary of the EDA we define a criteria to incomplete series 
starting_instalment_number= installments_payment_df.groupby("id_prev")["num_instalment_number"].transform("first")
starting_date= installments_payment_df.groupby("id_prev")["days_instalment"].transform("first")

installments_payment_df["is_potentially_incomplete_sequence"] = ((starting_instalment_number >  1)  & (starting_date < -2890 ))

In [ ]:
dtale.show(installments_payment_df [installments_payment_df["dead_tail_length"] !=0 ])

In [ ]:
potentially_on_going_id = installments_payment_df[installments_payment_df["days_instalment"] > (- 33)]["id_prev"].unique()
installments_payment_df["potentially_on_going"]= installments_payment_df["id_prev"].isin(potentially_on_going_id)   

In [ ]:
installments_payment_df["diff_expected_received"]= installments_payment_df["amt_instalment"] -  installments_payment_df["amt_payment"]
installments_payment_df["diff_deadline_factical_payment"]= installments_payment_df["days_instalment"] -  installments_payment_df["days_entry_payment"]
installments_payment_df["days_of_delinquency"]= installments_payment_df["diff_deadline_factical_payment"].clip(lower=0)
installments_payment_df["is_delinquency"] = installments_payment_df["days_of_delinquency"] > 0 

In [ ]:
installments_payment_df["is_underpayment"]= (installments_payment_df["amt_instalment"] >  installments_payment_df["amt_payment"]) & (installments_payment_df["amt_payment"] != 0)
rows_with_underpayment= installments_payment_df [installments_payment_df["is_underpayment"] == True]
installments_payment_df["days_of_underpayment"] = np.where(installments_payment_df["is_underpayment"], installments_payment_df["days_instalment"], np.nan)


In [ ]:
next_installment_number = installments_payment_df.groupby("id_prev")["num_instalment_number"].shift(-1)
next_version_number = installments_payment_df.groupby("id_prev")["num_instalment_version"].shift(-1)
repeated_installment_mask= (installments_payment_df ["num_instalment_number"] == next_installment_number)
underpayment_mask= (installments_payment_df[ "amt_payment" ] < installments_payment_df ["amt_instalment"]) & (installments_payment_df[ "amt_payment" ] == 0)
full_payment_mask= installments_payment_df[ "amt_payment" ] == installments_payment_df ["amt_instalment"] 
installments_payment_df[ "repeated_for_underpayment" ] = (repeated_installment_mask) & (underpayment_mask)
installments_payment_df[ "repeated_for_reschedule" ] = (repeated_installment_mask) & (full_payment_mask)
installments_payment_df["repeated_for_payment_in_advance"] = (repeated_installment_mask) & (installments_payment_df[ "amt_payment" ] == 0) & (installments_payment_df["days_of_delinquency"] == 0)
installments_payment_df["log_amt_instalment"]= np.log1p(installments_payment_df ["amt_instalment"] )
installments_payment_df["log_amt_payment"]= np.log1p(installments_payment_df ["amt_payment"] )

In [ ]:
installments_payment_df["extra_instalament"] = (installments_payment_df["amount_of_versions_in_sequence"] > 1) & (installments_payment_df["raw_size_serie"] <95)  & (installments_payment_df ["num_instalment_number"] >= 100)
interesting_cases= installments_payment_df[installments_payment_df["extra_instalament"] == True ]
dtale.show(installments_payment_df.head(500))

In [ ]:
installments_payment_df.head()

In [3]:
agg_metrics_df= installments_payment_df.groupby("id_prev").agg({

    #static values calculated for the entire squenece
    "raw_size_serie" : ["first"],
    "dead_tail_length" : ["first"],
    "amount_of_versions_in_sequence" : ["first"],
    "is_potentially_incomplete_sequence" : ["first"],
    "potentially_on_going" : ["first"],

    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_instalment": ["mean","std"],   
    "log_amt_payment": ["mean","std"],

    #natural scale
    "amt_instalment": ["max", "min","median","sum"],
    "amt_payment": ["max", "min","median","sum"],

    #computed_differences
    "diff_expected_received": ["max", "min","median","sum"],


    #categoricals
    "repeated_for_underpayment": ["mean","sum"],
    #"repeated_for_reschedule": ["mean","sum"],
    #"repeated_for_payment_in_advance": ["mean","sum"],
    "is_delinquency" : ["mean","sum"],


    #counters
    "days_of_delinquency":["mean","max","sum"],
    "days_of_underpayment" : ["mean","max"]
})

agg_metrics_df.columns = [
    f"instalments_{col[0]}" if col[1] == "first" else f"instalments_{col[0]}_{col[1]}"
    for col in agg_metrics_df.columns
]

agg_metrics_df = agg_metrics_df.reset_index()

agg_metrics_df["instalments_completion_ratio"] = np.where( agg_metrics_df["instalments_amt_instalment_sum"] > 0, agg_metrics_df["instalments_amt_payment_sum"] / agg_metrics_df["instalments_amt_instalment_sum"], 1.0 )  #if the debt is 0 or negative we assume competitud (1)                                                               )



KeyError: "Column(s) ['amount_of_versions_in_sequence', 'days_of_delinquency', 'days_of_underpayment', 'dead_tail_length', 'diff_expected_received', 'is_delinquency', 'is_potentially_incomplete_sequence', 'log_amt_instalment', 'log_amt_payment', 'potentially_on_going', 'raw_size_serie', 'repeated_for_underpayment'] do not exist"

In [ ]:
agg_metrics_df.to_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed.parquet")
agg_metrics_df.head()


In [28]:
installments_payment_df = pd.read_parquet(cfg.CLEANS_DIR / "installments_payments.train-cleaned.parquet")

#we starting catching this because we are probably cutting some parts of the temporal sequence
installments_payment_df["raw_size_serie"]= installments_payment_df.groupby("id_prev").transform("size")
installments_payment_df["amount_of_versions_in_sequence"] = installments_payment_df.groupby("id_prev")["num_instalment_version"].transform("nunique")

#we gonna use the knowlege recolected in the EDA. 
#For more details look eda_installments_payments.ipynb decisions summary 1#

#creating auxiliar columns
installments_payment_df["next_payment_value"] = installments_payment_df.groupby("id_prev")["days_entry_payment"].shift(-1)
nan_amount= installments_payment_df.groupby("id_prev")["days_entry_payment_is_missing"].transform("sum")

#defining the mask to separate the differents cases of missings values 
have_missing_entry_payment_mask= (installments_payment_df["next_payment_value"].isna())
not_a_deadtail_mask= (installments_payment_df["days_entry_payment"].isna() ) & ( installments_payment_df["next_payment_value"].notna())

#we want to catch the cases of  dead-tail so we starting filtering that cases with nans but that are not dead tails
non_dead_tail_nans= installments_payment_df[not_a_deadtail_mask]
ids_with_nulls_that_are_non_deadtails= non_dead_tail_nans["id_prev"].unique()
excluding_nans_non_deadtails_mask= ~(installments_payment_df["id_prev"].isin(ids_with_nulls_that_are_non_deadtails))
series_without_problematic_nans = installments_payment_df[excluding_nans_non_deadtails_mask]

#now this ID are series where the nans are deadtails, so count nans in "days_entry_payment" o "amt_payment" is calculate
#the lenght of the deadtail and we save that value in a new column
ids_with_deadtails= series_without_problematic_nans[series_without_problematic_nans["days_entry_payment"].isna()]["id_prev"].unique()
installments_payment_df["dead_tail_length"]= np.where(installments_payment_df["id_prev"].isin(ids_with_deadtails),  nan_amount, 0)

#also in the decision summary of the EDA we define a criteria to incomplete series 
starting_instalment_number= installments_payment_df.groupby("id_prev")["num_instalment_number"].transform("first")
starting_date= installments_payment_df.groupby("id_prev")["days_instalment"].transform("first")

installments_payment_df["is_potentially_incomplete_sequence"] = ((starting_instalment_number >  1)  & (starting_date < -2890 ))

potentially_on_going_id = installments_payment_df[installments_payment_df["days_instalment"] > (- 33)]["id_prev"].unique()
installments_payment_df["potentially_on_going"]= installments_payment_df["id_prev"].isin(potentially_on_going_id)   

installments_payment_df["is_underpayment"]= (installments_payment_df["amt_instalment"] >  installments_payment_df["amt_payment"]) & (installments_payment_df["amt_payment"] != 0)
rows_with_underpayment= installments_payment_df [installments_payment_df["is_underpayment"] == True]
installments_payment_df["days_of_underpayment"] = np.where(installments_payment_df["is_underpayment"], installments_payment_df["days_instalment"], np.nan)

installments_payment_df["diff_expected_received"]= installments_payment_df["amt_instalment"] -  installments_payment_df["amt_payment"]
installments_payment_df["diff_deadline_factical_payment"]= installments_payment_df["days_entry_payment"] - installments_payment_df["days_instalment"] 
installments_payment_df["days_in_advance"] = installments_payment_df["days_instalment"] - installments_payment_df["days_entry_payment"]
installments_payment_df["days_of_delinquency"]= installments_payment_df["diff_deadline_factical_payment"].clip(lower=0)
installments_payment_df["days_in_advance"]= installments_payment_df["days_in_advance"].clip(lower=0)
installments_payment_df["is_delinquency"] = installments_payment_df["days_of_delinquency"] > 0 




#installments_payment_df["payment_ratio"] = np.where(
 #   installments_payment_df["amt_instalment"] > 0 ,installments_payment_df["amt_payment"] / installments_payment_df["amt_instalment"], 1)

#installments_payment_df["mean_historical_payment_rate"] = installments_payment_df.groupby("id_curr")["payment_ratio"].transform("mean")


#last_year= installments_payment_df[installments_payment_df[column_order_reference] > -365]

#avg_last_year= last_year.groupby("id_curr")["payment_ratio"].mean()

#installments_payment_df["mean_ratio_last_year"] = installments_payment_df["id_curr"].map(avg_last_year)


#last_two_years= installments_payment_df[installments_payment_df[column_order_reference] > -720]

#avg_last_two_years= last_two_years.groupby("id_curr")["payment_ratio"].mean()

#installments_payment_df["mean_ratio_last_two_years"] = installments_payment_df["id_curr"].map(avg_last_two_years)

#installments_payment_df["payment_tendence"] = installments_payment_df["mean_ratio_last_year"] - installments_payment_df["mean_ratio_last_two_years"] 





next_installment_number = installments_payment_df.groupby("id_prev")["num_instalment_number"].shift(-1)
next_version_number = installments_payment_df.groupby("id_prev")["num_instalment_version"].shift(-1)
repeated_installment_mask= (installments_payment_df ["num_instalment_number"] == next_installment_number)
underpayment_mask= (installments_payment_df[ "amt_payment" ] < installments_payment_df ["amt_instalment"])
full_payment_mask= installments_payment_df[ "amt_payment" ] == installments_payment_df ["amt_instalment"] 
installments_payment_df[ "repeated_for_underpayment" ] = (repeated_installment_mask) & (underpayment_mask)
#installments_payment_df[ "repeated_for_reschedule" ] = (repeated_installment_mask) & (full_payment_mask)
#installments_payment_df["repeated_for_payment_in_advance"] = (repeated_installment_mask) & (installments_payment_df[ "amt_payment" ] == 0) & (installments_payment_df["days_of_delinquency"] == 0)
installments_payment_df["log_amt_instalment"]= np.log1p(installments_payment_df ["amt_instalment"] )
installments_payment_df["log_amt_payment"]= np.log1p(installments_payment_df ["amt_payment"] )

installments_payment_df["extra_instalament"] = (installments_payment_df["amount_of_versions_in_sequence"] > 1) & (installments_payment_df["raw_size_serie"] <95)  & (installments_payment_df ["num_instalment_number"] >= 100)
interesting_cases= installments_payment_df[installments_payment_df["extra_instalament"] == True ]

agg_metrics_df= installments_payment_df.groupby("id_prev").agg({

    #static values calculated for the entire squenece
    #"id_curr" : ["first"],
    "raw_size_serie" : ["first"],
    "dead_tail_length" : ["first"],
    "potentially_on_going" : ["first"],
    "amount_of_versions_in_sequence" : ["first"],
    #"mean_ratio_last_year" :["first"],
    #"mean_ratio_last_two_years" :["first"],
    #"mean_historical_payment_rate": ["first"],
    #"payment_tendence" :["first"],

    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_instalment": ["mean","std"],   
    "log_amt_payment": ["mean","std"],

    #natural scale
    "amt_instalment": ["max", "min","median","sum"], #, "count"
    "amt_payment": ["max", "min","median","sum"],
    "extra_instalament":["sum","mean"],
   

    #computed_differences
    "diff_expected_received": ["max", "min", "median", "sum"],
    
    #categoricals
    "repeated_for_underpayment": ["mean","sum"],
    #"repeated_for_reschedule": ["mean","sum"],
    "is_delinquency" : ["mean","sum"],


    #counters
    #"days_instalment":["min"],
    "days_of_delinquency":["mean","max","sum"],
    "days_in_advance":["mean","max","sum"],
    "days_of_underpayment" : ["max"]
})

agg_metrics_df.columns = [
    f"instalments_{col[0]}" if col[1] == "first" else f"instalments_{col[0]}_{col[1]}"
    for col in agg_metrics_df.columns
]

agg_metrics_df = agg_metrics_df.reset_index()

agg_metrics_df["instalments_completion_ratio"] = np.where( agg_metrics_df["instalments_amt_instalment_sum"] > 0, agg_metrics_df["instalments_amt_payment_sum"] / agg_metrics_df["instalments_amt_instalment_sum"], 1.0 )  #if the debt is 0 or negative we assume competitud (1)   

agg_metrics_df.to_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")

agg_metrics_last_three_months= get_time_window_metrics(-90, installments_payment_df)
agg_metrics_last_year = get_time_window_metrics(-365, installments_payment_df)

temporal_windows_df = agg_metrics_last_year.merge(
    agg_metrics_last_three_months, 
    on="id_curr", 
    how="outer",  
)

temporal_windows_df["payment_trend"] = np.where(temporal_windows_df["last_90_instalments_completion_ratio"] != 0 , temporal_windows_df["last_365_instalments_completion_ratio"] / temporal_windows_df["last_90_instalments_completion_ratio"],np.nan)
temporal_windows_df["delincuency_trend"] = temporal_windows_df["last_365_instalments_days_of_delinquency_mean"] - temporal_windows_df["last_90_instalments_days_of_delinquency_mean"] 
temporal_windows_df["underpayment_trend"] = temporal_windows_df["last_365_instalments_is_underpayment_mean"] - temporal_windows_df["last_90_instalments_is_underpayment_mean"]

temporal_windows_df_to_save=pd.DataFrame()
#temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_sum"]= temporal_windows_df["last_365_instalments_days_of_delinquency_sum"]
#temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_mean"]= temporal_windows_df["last_365_instalments_days_of_delinquency_mean"]
#temporal_windows_df_to_save["delincuency_trend"] = temporal_windows_df["delincuency_trend"]
temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_sum"]= temporal_windows_df["last_365_instalments_days_of_delinquency_sum"]
temporal_windows_df_to_save["last_365_instalments_days_of_delinquency_mean"]= temporal_windows_df["last_365_instalments_days_of_delinquency_mean"]
temporal_windows_df_to_save["last_365_instalments_extra_instalament_mean"]= temporal_windows_df["last_365_instalments_extra_instalament_mean"]
temporal_windows_df_to_save["last_365_instalments_amt_payment_min"]= temporal_windows_df["last_365_instalments_amt_payment_min"]
temporal_windows_df_to_save["last_90_instalments_amt_instalment_min"]= temporal_windows_df["last_90_instalments_amt_instalment_min"]
temporal_windows_df_to_save["last_365_instalments_is_delinquency_mean"]= temporal_windows_df["last_365_instalments_is_delinquency_mean"]
temporal_windows_df_to_save["last_365_instalments_completion_ratio"]= temporal_windows_df["last_365_instalments_completion_ratio"] 
temporal_windows_df_to_save["last_90_instalments_completion_ratio"]= temporal_windows_df["last_90_instalments_completion_ratio"] 
temporal_windows_df_to_save["last_90_instalments_amt_instalment_min"]= temporal_windows_df["last_90_instalments_amt_instalment_min"]
temporal_windows_df_to_save["payment_trend"]= temporal_windows_df["payment_trend"]


temporal_windows_df_to_save["id_curr"]= temporal_windows_df["id_curr"]
temporal_windows_df_to_save.to_parquet(cfg.PROCESSED_DIR / "time_window_instalments.parquet")






In [8]:
dtale.show(agg_metrics_last_year.head())

2026-06-21 17:10:33,668 - ERROR    - Exception on /health [GET]
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 1821, in full_dispatch_request
    rv = self.preprocess_request()
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\flask\app.py", line 2313, in preprocess_request
    rv = self.ensure_sync(before_func)()
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\defa

In [ ]:
last_three_months_agg

,id_curr,last_year_instalments_amt_instalment_max,last_year_instalments_amt_instalment_min,last_year_instalments_amt_instalment_median,last_year_instalments_amt_instalment_sum,last_year_instalments_amt_payment_max,last_year_instalments_amt_payment_min,last_year_instalments_amt_payment_median,last_year_instalments_amt_payment_sum,last_year_instalments_days_of_delinquency_mean,...,last_year_instalments_days_of_underpayment_max,last_year_instalments_diff_expected_received_max,last_year_instalments_diff_expected_received_min,last_year_instalments_diff_expected_received_median,last_year_instalments_diff_expected_received_sum,last_year_instalments_is_delinquency_mean,last_year_instalments_is_delinquency_sum,last_year_instalments_extra_instalament_sum,last_year_instalments_extra_instalament_mean,last_year_instalments_completion_ratio
0,100002,53093.745,9251.775,9251.775,154863.270,53093.745,9251.775,9251.775,154863.270,0.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0,0,0.0,1.0
1,100006,691786.890,29027.520,29027.520,982062.090,691786.890,29027.520,29027.520,982062.090,0.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0,0,0.0,1.0
2,100007,16046.100,16037.640,16037.640,208497.780,16046.100,16037.640,16037.640,208497.780,0.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0,0,0.0,1.0
3,100008,17885.835,17876.115,17885.835,178848.630,17885.835,17876.115,17885.835,178848.630,0.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0,0,0.0,1.0
4,100009,10418.670,7499.565,8821.260,198310.545,10418.670,7499.565,8821.260,198310.545,0.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0,0,0.0,1.0


In [6]:
df1= pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")
df2 = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")


# 1. ¿Qué columnas tiene df1 que le faltan a df2?
solo_en_df1 = set(df1.columns) - set(df2.columns)
print(f"Columnas solo en el primer DF ({len(solo_en_df1)}):", solo_en_df1)

# 2. ¿Qué columnas tiene df2 que le faltan a df1?
solo_en_df2 = set(df2.columns) - set(df1.columns)
print(f"Columnas solo en el segundo DF ({len(solo_en_df2)}):", solo_en_df2)

# 3. (Opcional) Ver ABSOLUTAMENTE TODAS las columnas que no coinciden entre ambos
diferencia_total = set(df1.columns) ^ set(df2.columns) 
print("Todas las columnas diferentes:", diferencia_total)

Columnas solo en el primer DF (0): set()
Columnas solo en el segundo DF (5): {'instalments_amt_instalment_count_prev_1', 'instalments_amt_instalment_count_prev_3', 'log_diff_application_credit_std', 'instalments_amt_instalment_count_prev_2', 'diff_application_credit_std'}
Todas las columnas diferentes: {'log_diff_application_credit_std', 'instalments_amt_instalment_count_prev_3', 'instalments_amt_instalment_count_prev_2', 'instalments_amt_instalment_count_prev_1', 'diff_application_credit_std'}
